# Multimodal AI — Blood Work Analysis

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

e:\AamadNaseem\MSAIEngineering\AI Engineer\AiProjects\agentic-ai-crash-course\agentic-ai-crashcourse-code\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

## Encode the image and send to the vision model

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

llm = ChatGroq(model="qwen/qwen3.8-27b")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

Based on the blood work report provided, here is a breakdown of all the test results and flags for any values that are outside the normal range.

### **Patient Information**
- **Name:** Sohail Khaliqu
- **Age:** 48
- **Gender:** Male
- **Date:** May 7, 2026

---

### **Complete Blood Count (CBC)**
- **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5)
- **Hematocrit:** 44% (Normal: 41-53%)
- **WBC:** 6.8 x10^3/uL (Normal: 4.5-11.0)
- **Platelets:** 220 x10^3/uL (Normal: 150-400)

### **Lipid Panel**
- **Total Cholesterol:** 238 mg/dL (Normal: <200) ⚠️ **High**
- **LDL Cholesterol:** 162 mg/dL (Normal: <100) ⚠️ **High**
- **HDL Cholesterol:** 36 mg/dL (Normal: >40) ⚠️ **Low**
- **Triglycerides:** 188 mg/dL (Normal: <150) ⚠️ **High**

### **Metabolic Panel**
- **Glucose (Fasting):** 92 mg/dL (Normal: 70-99)
- **HbA1c:** 5.3% (Normal: <5.7%)
- **Creatinine:** 1.0 mg/dL (Normal: 0.7-1.3)
- **eGFR:** 82 mL/min (Normal: >60)

### **Liver Function**
- **ALT:** 28 U/L (Normal: 7-40)
- **AST:** 25 U/L

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [3]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [4]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [5]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

**Diet Plan for High Cholesterol**

Based on the recommendation, here is the suggested diet plan:

**✅ Foods to Eat:**
*   Fruits
*   Vegetables
*   Whole grains
*   Lean protein

**❌ Foods to Avoid:**
*   Red meat
*   Fried food
*   Full-fat dairy
*   Processed snacks
